# 02 · Embeddings — the common base of Pebble's brain

A **sentence encoder** turns any sentence into a vector (here 384 numbers) so that sentences with the
same *meaning* land close together — across languages and scripts. One encoder pass can then feed
many tiny "heads" (intent, mood, memory search…). That's the hybrid architecture in our plan.

We use `intfloat/multilingual-e5-small` (MIT licence, ~118M params) as the **teacher**. In M1 we'll
distil it into a smaller student that's fast on an 8 GB laptop.

First run downloads ~470 MB of weights (cached afterwards).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("intfloat/multilingual-e5-small", device=device)
# e5 models expect a "query: " prefix for short texts.
embed = lambda texts: encoder.encode([f"query: {t}" for t in texts], normalize_embeddings=True, convert_to_tensor=True)

## 1. Meaning, not spelling
Cosine similarity of normalised vectors = dot product. 1.0 = same meaning, ~0.7 = unrelated for e5.

In [ ]:
sentences = [
    "remind me tomorrow at 5",
    "kal 5 baje yaad dila dena",
    "कल 5 बजे याद दिला देना",
    "I drank a glass of water",
    "paani pi liya",
    "my exam is next week",
]
E = embed(sentences)
sim = (E @ E.T).cpu()
import pandas as pd
pd.DataFrame(sim.numpy(), index=sentences, columns=[s[:14] + "…" for s in sentences]).round(2)

Look at the first three rows: English, Roman Hinglish and Devanagari land close together. The
character-hash features from notebook 01 could never do that.

## 2. Same tiny head, now on embeddings (transfer learning)
Re-use notebook 01's dataset, but feed the head encoder vectors instead of character hashes.

In [ ]:
from torch import nn

data = [
    ("remind me to call mom at 7pm", "remind"), ("kal 5 baje DBMS assignment yaad dila dena", "remind"),
    ("शाम को दवा लेने की याद दिलाना", "remind"), ("remind me tomorrow about the meeting", "remind"),
    ("mujhe 10 minute baad yaad dilana", "remind"),
    ("ek glass paani pi liya", "water"), ("drank 2 glasses of water", "water"), ("मैंने पानी पी लिया", "water"),
    ("+1 water", "water"), ("paani piya abhi", "water"),
    ("note: idea for the pet climbing windows", "note"), ("yeh likh lo: grocery mein doodh lena hai", "note"),
    ("नोट करो कल की क्लास कैंसिल है", "note"), ("save a note about the project deadline", "note"),
    ("likh ke rakh lo ki wifi password badalna hai", "note"),
]
labels = sorted({y for _, y in data})
X = embed([t for t, _ in data])
y = torch.tensor([labels.index(l) for _, l in data], device=device)

torch.manual_seed(0)
head = nn.Linear(X.shape[1], len(labels)).to(device)   # a head can be THIS small
opt = torch.optim.AdamW(head.parameters(), lr=5e-2)
for step in range(150):
    loss = nn.functional.cross_entropy(head(X), y)
    opt.zero_grad(); loss.backward(); opt.step()
print("final loss", round(loss.item(), 4))

In [ ]:
tests = [
    ("yaad dilana ki 6 baje gym jaana hai", "remind"), ("I just had some water", "water"),
    ("पानी की बोतल खत्म", "water"), ("jot this down: buy a new charger", "note"),
    ("remind me to drink water at 4", "remind"),
]
with torch.no_grad():
    P = head(embed([t for t, _ in tests])).softmax(-1)
for (text, want), p in zip(tests, P):
    got = labels[p.argmax().item()]
    print(f"{'✓' if got == want else '✗'} {text!r:45} → {got} ({p.max().item():.0%})")

## 3. Preview: few-shot "teach Pebble a new command" (meta-learning, simplest form)
**Prototypical networks:** average the embeddings of 3 examples → that's the new intent's prototype.
Classify by nearest prototype. No training at all — this is how Pebble will learn new commands
on a low-spec laptop in milliseconds.

In [ ]:
new_intent = ["start a 25 minute focus timer", "pomodoro shuru karo", "फोकस टाइमर चालू करो"]
prototypes = {l: X[y == i].mean(0) for i, l in enumerate(labels)}
prototypes["focus"] = embed(new_intent).mean(0)

def classify(text):
    e = embed([text])[0]
    scores = {l: float(e @ p / p.norm()) for l, p in prototypes.items()}
    return max(scores, key=scores.get), scores

for t in ["focus mode on karo for 30 min", "remind me at 9", "थोड़ा पानी पिया"]:
    print(t, "→", classify(t)[0])

## What you just learned
* **Embeddings / transfer learning:** a pretrained encoder carries meaning across Hindi, Hinglish
  and English; a one-layer head is enough on top.
* **The hybrid design:** one encoder pass, many cheap heads.
* **Few-shot prototypes:** adding an intent without training.

**Next — notebook 03 (M1):** real data (Amazon MASSIVE hi-IN/en-US + your recorded commands), a held-out
eval set, slot extraction (times, dates), then distilling the encoder small enough for 8 GB laptops
and exporting to ONNX for the Kotlin app.